In [1]:
import os
import time
import requests

# CONFIGURATION
# We will fetch UK Public General Acts (ukpga)
YEARS = [2023, 2024] 
MAX_ACTS_PER_YEAR = 100 # Adjust to get your ~200 total
OUTPUT_DIR = "data/raw_legislation"

os.makedirs(OUTPUT_DIR, exist_ok=True)

headers = {
    'User-Agent': 'LegalKGent-Research-Project/1.0 (contact@university.edu)'
}

def download_act(year, number):
    # Legislation.gov.uk exposes XML via /data.xml endpoint
    url = f"https://www.legislation.gov.uk/ukpga/{year}/{number}/data.xml"
    
    try:
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            filename = f"ukpga_{year}_{number}.xml"
            filepath = os.path.join(OUTPUT_DIR, filename)
            
            with open(filepath, "wb") as f:
                f.write(response.content)
            print(f"[SUCCESS] Downloaded: {filename}")
            return True
        elif response.status_code == 404:
            print(f"[MISSING] Act {number} of {year} does not exist.")
            return False
        else:
            print(f"[ERROR] HTTP {response.status_code} for {url}")
            return False
            
    except Exception as e:
        print(f"[ERROR] Failed to fetch {url}: {e}")
        return False

# MAIN LOOP
total_downloaded = 0

for year in YEARS:
    print(f"--- Fetching Acts for {year} ---")
    consecutive_failures = 0
    
    for number in range(1, MAX_ACTS_PER_YEAR + 1):
        success = download_act(year, number)
        
        if success:
            total_downloaded += 1
            consecutive_failures = 0
        else:
            consecutive_failures += 1
        
        # Polite delay to respect server load
        time.sleep(1) 
        
        # Stop if we hit 10 consecutive missing acts (end of year likely reached)
        if consecutive_failures >= 10:
            print(f"Stopping {year} loop after 10 failures.")
            break

print(f"\nDone! Total files downloaded: {total_downloaded}")
print(f"Files saved to: {OUTPUT_DIR}")

--- Fetching Acts for 2023 ---
[SUCCESS] Downloaded: ukpga_2023_1.xml
[SUCCESS] Downloaded: ukpga_2023_2.xml
[SUCCESS] Downloaded: ukpga_2023_3.xml
[SUCCESS] Downloaded: ukpga_2023_4.xml
[SUCCESS] Downloaded: ukpga_2023_5.xml
[SUCCESS] Downloaded: ukpga_2023_6.xml
[SUCCESS] Downloaded: ukpga_2023_7.xml
[SUCCESS] Downloaded: ukpga_2023_8.xml
[SUCCESS] Downloaded: ukpga_2023_9.xml
[SUCCESS] Downloaded: ukpga_2023_10.xml
[SUCCESS] Downloaded: ukpga_2023_11.xml
[SUCCESS] Downloaded: ukpga_2023_12.xml
[SUCCESS] Downloaded: ukpga_2023_13.xml
[SUCCESS] Downloaded: ukpga_2023_14.xml
[SUCCESS] Downloaded: ukpga_2023_15.xml
[SUCCESS] Downloaded: ukpga_2023_16.xml
[SUCCESS] Downloaded: ukpga_2023_17.xml
[SUCCESS] Downloaded: ukpga_2023_18.xml
[SUCCESS] Downloaded: ukpga_2023_19.xml
[SUCCESS] Downloaded: ukpga_2023_20.xml
[SUCCESS] Downloaded: ukpga_2023_21.xml
[SUCCESS] Downloaded: ukpga_2023_22.xml
[SUCCESS] Downloaded: ukpga_2023_23.xml
[SUCCESS] Downloaded: ukpga_2023_24.xml
[SUCCESS] Download

In [2]:
import os
import time
import requests

# CONFIGURATION
# We added EWHC (High Court) because it has way more volume than UKSC
COURTS = ["uksc", "ewca/civ", "ewhc/admin", "ewhc/ch"] 
YEARS = [2023, 2024]
MAX_PER_COURT = 15  # 15 * 4 courts * 2 years = ~120 files
OUTPUT_DIR = "data/raw_caselaw"

os.makedirs(OUTPUT_DIR, exist_ok=True)

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/xml, text/xml, */*'
}

def try_download(url, filename):
    try:
        r = requests.get(url, headers=headers, timeout=10)
        # Check if we actually got XML (start with <) and not an HTML error page
        if r.status_code == 200 and r.content.strip().startswith(b"<"):
            with open(filename, "wb") as f:
                f.write(r.content)
            return True
    except Exception as e:
        pass
    return False

def download_case(court, year, number):
    base_filename = os.path.join(OUTPUT_DIR, f"{court.replace('/', '_')}_{year}_{number}.xml")
    
    # URL STRATEGY: Try the 3 known patterns for TNA XML
    # Pattern 1: Direct XML endpoint (Most reliable for new system)
    url1 = f"https://caselaw.nationalarchives.gov.uk/{court}/{year}/{number}/xml"
    
    # Pattern 2: Old style .xml extension
    url2 = f"https://caselaw.nationalarchives.gov.uk/{court}/{year}/{number}.xml"
    
    # Pattern 3: Data endpoint
    url3 = f"https://caselaw.nationalarchives.gov.uk/{court}/{year}/{number}/data.xml"

    if try_download(url1, base_filename): return "Method 1"
    if try_download(url2, base_filename): return "Method 2"
    if try_download(url3, base_filename): return "Method 3"
    
    return None

# MAIN LOOP
print(f"Starting Robust Download... Saving to {OUTPUT_DIR}")
total_success = 0

for court in COURTS:
    for year in YEARS:
        fails = 0
        print(f"\n--- {court.upper()} {year} ---")
        for num in range(1, MAX_PER_COURT + 1):
            method = download_case(court, year, num)
            
            if method:
                print(f"[OK] {court} {year}/{num} (via {method})")
                total_success += 1
                fails = 0
            else:
                print(f"[x] {court} {year}/{num} missing.")
                fails += 1
            
            # Smart backoff: If we miss 5 in a row, likely end of sequence for this court/year
            if fails >= 5:
                print(">> Too many 404s, skipping to next court/year...")
                break
                
            time.sleep(1) # Polite delay

print(f"\nDownload Complete. Total Files: {total_success}")

Starting Robust Download... Saving to data/raw_caselaw

--- UKSC 2023 ---
[OK] uksc 2023/1 (via Method 3)
[OK] uksc 2023/2 (via Method 3)
[OK] uksc 2023/3 (via Method 3)
[OK] uksc 2023/4 (via Method 3)
[OK] uksc 2023/5 (via Method 3)
[OK] uksc 2023/6 (via Method 3)
[OK] uksc 2023/7 (via Method 3)
[OK] uksc 2023/8 (via Method 3)
[OK] uksc 2023/9 (via Method 3)
[OK] uksc 2023/10 (via Method 3)
[OK] uksc 2023/11 (via Method 3)
[OK] uksc 2023/12 (via Method 3)
[OK] uksc 2023/13 (via Method 3)
[OK] uksc 2023/14 (via Method 3)
[OK] uksc 2023/15 (via Method 3)

--- UKSC 2024 ---
[OK] uksc 2024/1 (via Method 3)
[OK] uksc 2024/2 (via Method 3)
[OK] uksc 2024/3 (via Method 3)
[OK] uksc 2024/4 (via Method 3)
[OK] uksc 2024/5 (via Method 3)
[OK] uksc 2024/6 (via Method 3)
[OK] uksc 2024/7 (via Method 3)
[OK] uksc 2024/8 (via Method 3)
[OK] uksc 2024/9 (via Method 3)
[OK] uksc 2024/10 (via Method 3)
[OK] uksc 2024/11 (via Method 3)
[OK] uksc 2024/12 (via Method 3)
[OK] uksc 2024/13 (via Method 3)
[

In [5]:
import os
import json
import re
import xml.etree.ElementTree as ET

# CONFIG
INPUT_ACTS = "data/raw_legislation"
INPUT_CASES = "data/raw_caselaw"
OUTPUT_FILE = "data/legal_corpus_clean.json"

# NAMESPACE HANDLING
# XML tags often look like {http://www.legislation.gov.uk/namespaces/legislation}P1
# We strip the {...} part to make logic simple.
def strip_ns(tag):
    if '}' in tag:
        return tag.split('}', 1)[1]
    return tag

def clean_text(element):
    """
    Recursively extract text with proper spacing.
    Handles the '1For' problem by adding space after tags.
    """
    text = ""
    if element.text:
        text += element.text.strip() + " "
    
    for child in element:
        # Recursively get child text
        child_text = clean_text(child)
        text += child_text
        
        # Add tail text (text appearing after a closing tag)
        if child.tail:
            text += child.tail.strip() + " "
            
    return re.sub(r'\s+', ' ', text).strip() # Collapse multiple spaces

def process_legislation(filepath):
    chunks = []
    filename = os.path.basename(filepath)
    
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
        
        # 1. Extract Global Metadata (Title)
        act_title = "Unknown Act"
        for elem in root.iter():
            if strip_ns(elem.tag) == 'title':
                act_title = elem.text
                break
        
        # 2. Extract Sections (P1 tags)
        # We look for P1 (Section) and P1group (Section with Title)
        for p1 in root.iter():
            if strip_ns(p1.tag) == 'P1':
                
                # Try to find the Section Number
                number_tag = p1.find(f".//{{*}}Pnumber")
                sec_num = number_tag.text if number_tag is not None else "Unknown"
                
                # Try to find the Title (often in a sibling or parent P1group)
                # For MVP, we'll stick to the Act Title + Section Number
                
                # Clean the Text
                # We specifically target 'P1para' to avoid grabbing the number twice
                para_tag = p1.find(f".//{{*}}P1para")
                if para_tag is not None:
                    raw_text = clean_text(para_tag)
                else:
                    raw_text = clean_text(p1) # Fallback

                chunk = {
                    "id": f"{filename}_s{sec_num}",
                    "source": "legislation",
                    "doc_title": act_title,
                    "section": sec_num,
                    "content": f"ACT: {act_title} | SECTION: {sec_num} | TEXT: {raw_text}"
                }
                chunks.append(chunk)
                
    except Exception as e:
        print(f"Error parsing {filename}: {e}")
        
    return chunks

def process_caselaw(filepath):
    chunks = []
    filename = os.path.basename(filepath)
    
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
        
        # 1. Metadata (Case Name & Date)
        case_name = "Unknown Case"
        case_date = "Unknown Date"
        
        for work in root.iter():
            tag = strip_ns(work.tag)
            if tag == 'FRBRname':
                case_name = work.get('value')
            if tag == 'FRBRdate' and work.get('name') == 'judgment':
                case_date = work.get('date')
        
        # 2. Extract Paragraphs
        for para in root.iter():
            if strip_ns(para.tag) == 'paragraph':
                # Get Paragraph ID
                p_id = "para_?"
                num_tag = para.find(f".//{{*}}num")
                if num_tag is not None:
                    p_id = num_tag.text.strip()
                
                # Get Text
                content_tag = para.find(f".//{{*}}content")
                if content_tag is not None:
                    text = clean_text(content_tag)
                    
                    # Create Chunk
                    chunk = {
                        "id": f"{filename}_{p_id}",
                        "source": "judgment",
                        "doc_title": case_name,
                        "date": case_date,
                        "para_id": p_id,
                        "content": f"CASE: {case_name} ({case_date}) | PARA: {p_id} | TEXT: {text}"
                    }
                    chunks.append(chunk)

    except Exception as e:
        print(f"Error parsing {filename}: {e}")
        
    return chunks

# MAIN RUNNER
if __name__ == "__main__":
    all_data = []
    
    # Process Acts
    print("Processing Legislation...")
    for f in os.listdir(INPUT_ACTS):
        if f.endswith(".xml"):
            path = os.path.join(INPUT_ACTS, f)
            all_data.extend(process_legislation(path))
            
    # Process Cases
    print("Processing Case Law...")
    for f in os.listdir(INPUT_CASES):
        if f.endswith(".xml"):
            path = os.path.join(INPUT_CASES, f)
            all_data.extend(process_caselaw(path))
            
    print(f"Total Chunks Generated: {len(all_data)}")
    
    # Save to JSON
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(all_data, f, indent=2, ensure_ascii=False)
    
    print(f"Saved cleaned corpus to {OUTPUT_FILE}")

Processing Legislation...
Processing Case Law...
Total Chunks Generated: 17496
Saved cleaned corpus to data/legal_corpus_clean.json


In [6]:
import json
import statistics
from collections import Counter

# CONFIG
INPUT_FILE = "data/legal_corpus_clean.json"

def validate_corpus(filepath):
    print(f"--- VALIDATING: {filepath} ---")
    
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        print(f"[FATAL] JSON is invalid/corrupt: {e}")
        return

    total_items = len(data)
    print(f"Total Chunks: {total_items}")

    # 1. Structural Validation (Schema Check)
    required_keys = ["id", "source", "content"]
    missing_keys_count = 0
    malformed_examples = []

    # 2. Content Analysis
    sources = Counter()
    lengths = []
    empty_content = 0
    
    # 3. Specific Logic Checks
    missing_titles = 0
    missing_dates = 0  # Critical for Case Law
    
    for i, item in enumerate(data):
        # Check Keys
        if not all(k in item for k in required_keys):
            missing_keys_count += 1
            if len(malformed_examples) < 3: 
                malformed_examples.append(item)
            continue

        # Check Source Distribution
        src = item.get("source", "UNKNOWN")
        sources[src] += 1
        
        # Check Content Length (Characters)
        text = item.get("content", "")
        if not text or len(text.strip()) < 10:  # <10 chars is likely junk
            empty_content += 1
        else:
            lengths.append(len(text))

        # Check Metadata Completeness
        if src == "legislation":
            if "doc_title" not in item or not item["doc_title"]:
                missing_titles += 1
        elif src == "judgment":
            if "date" not in item or not item["date"]:
                missing_dates += 1

    # --- REPORTING ---
    
    print("\n[1] SOURCE DISTRIBUTION:")
    for src, count in sources.items():
        print(f"  - {src.upper()}: {count} chunks ({count/total_items:.1%})")

    print("\n[2] TEXT LENGTH STATS (Characters):")
    if lengths:
        print(f"  - Min: {min(lengths)}")
        print(f"  - Max: {max(lengths)}")
        print(f"  - Avg: {int(statistics.mean(lengths))}")
        print(f"  - Median: {int(statistics.median(lengths))}")
    
    print("\n[3] DATA QUALITY ALERTS:")
    if missing_keys_count > 0:
        print(f"  [CRITICAL] Items missing keys: {missing_keys_count}")
        print(f"  Example: {malformed_examples[0]}")
    else:
        print("  [OK] All items have required keys.")

    if empty_content > 0:
        print(f"  [WARNING] 'Empty' content (<10 chars): {empty_content} chunks")
        print("  (These create 'ghost nodes' in the graph. Consider filtering them.)")
    else:
        print("  [OK] No empty content found.")

    if missing_titles > 0:
        print(f"  [WARNING] Acts missing Titles: {missing_titles}")
    
    if missing_dates > 0:
        print(f"  [CRITICAL] Cases missing Dates: {missing_dates}")
        print("  (Without dates, we cannot build the 'Hierarchy Logic' later.)")
    else:
        print("  [OK] All Cases have dates.")

# RUN
if __name__ == "__main__":
    validate_corpus(INPUT_FILE)

--- VALIDATING: data/legal_corpus_clean.json ---
Total Chunks: 17496

[1] SOURCE DISTRIBUTION:
  - LEGISLATION: 9827 chunks (56.2%)
  - JUDGMENT: 7669 chunks (43.8%)

[2] TEXT LENGTH STATS (Characters):
  - Min: 62
  - Max: 157181
  - Avg: 1076
  - Median: 696

[3] DATA QUALITY ALERTS:
  [OK] All items have required keys.
  [OK] No empty content found.
  [OK] All Cases have dates.


In [7]:
import json
import random

INPUT_FILE = "data/legal_corpus_clean.json"

def inspect_samples():
    try:
        with open(INPUT_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        print(f"Error loading file: {e}")
        return

    # separate by type
    legislation = [d for d in data if d.get("source") == "legislation"]
    judgments = [d for d in data if d.get("source") == "judgment"]
    
    print(f"--- TOTAL LOADED: {len(data)} ---")

    # 1. SHOW NORMAL LEGISLATION
    print("\n=== SAMPLE: LEGISLATION (Normal) ===")
    if legislation:
        sample = random.choice(legislation)
        print(json.dumps(sample, indent=2))
    else:
        print("No legislation found.")

    # 2. SHOW NORMAL JUDGMENT
    print("\n=== SAMPLE: JUDGMENT (Normal) ===")
    if judgments:
        sample = random.choice(judgments)
        print(json.dumps(sample, indent=2))
    else:
        print("No judgments found.")

    # 3. FIND THE "MONSTER" CHUNK
    print("\n=== ALERT: THE LARGEST CHUNK ===")
    # Find the item with the longest 'content' string
    monster = max(data, key=lambda x: len(x.get("content", "")))
    
    print(f"ID: {monster.get('id')}")
    print(f"Length: {len(monster.get('content', ''))} characters")
    print(f"Snippet (First 500 chars):\n{monster.get('content', '')[:500]}...")
    print("\n[ADVICE]: If this is a Table of Contents or a massive Schedule, you might want to split it further in 'preprocess.py'.")

if __name__ == "__main__":
    inspect_samples()

--- TOTAL LOADED: 17496 ---

=== SAMPLE: LEGISLATION (Normal) ===
{
  "id": "ukpga_2023_36.xml_sNone",
  "source": "legislation",
  "doc_title": "Social Housing (Regulation) Act 2023",
  "section": null,
  "content": "ACT: Social Housing (Regulation) Act 2023 | SECTION: None | TEXT: 1 The Housing and Regeneration Act 2008 is amended as follows. 2 After section 161 insert\u2014 Company: receipt of transfer of engagements from registered society 161A 1 This section applies to a registered provider which is a registered company. 2 The registered provider must notify the regulator if a registered society which is not a registered provider passes a resolution under section 112(1)(c) of the Co-operative and Community Benefit Societies Act 2014 transferring its engagements to the registered provider. 3 The Financial Conduct Authority may register the resolution only if the registered society which passed it has confirmed to the Financial Conduct Authority that the regulator has been notified.